In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score



In [3]:
# LOAD DATA
# =========================
df = pd.read_csv("sample_data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Drop ID column
df.drop(columns=["customerID"], inplace=True)

In [4]:
# Fix TotalCharges
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

# Encode target
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

In [6]:
# ONE HOT ENCODING

categorical_cols = [
    'gender','Partner','Dependents','PhoneService','MultipleLines',
    'InternetService','OnlineSecurity','OnlineBackup','DeviceProtection',
    'TechSupport','StreamingTV','StreamingMovies','Contract',
    'PaperlessBilling','PaymentMethod'
]

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [7]:
# Split features/target
X = df_encoded.drop("Churn", axis=1)
y = df_encoded["Churn"]

print("Final feature count:", X.shape[1])


Final feature count: 30


In [8]:
# SAVE FEATURE COLUMNS (CRITICAL)
# =========================
feature_columns = X.columns.tolist()
joblib.dump(feature_columns, "columns_onehot.pkl")


['columns_onehot.pkl']

In [10]:
# TRAIN TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [11]:
# MODELS
# =========================
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "SVM": SVC(probability=True)
}

In [12]:
# TRAIN + EVALUATE
# =========================
results = []

best_model = None
best_name = ""
best_auc = 0

In [13]:
for name, model in models.items():

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    cv_f1 = cross_val_score(pipe, X_train, y_train, cv=5, scoring="f1").mean()

    results.append([name, acc, prec, rec, f1, auc, cv_f1])

    if auc > best_auc:
        best_auc = auc
        best_model = pipe
        best_name = name



In [14]:
# RESULTS TABLE
# =========================
results_df = pd.DataFrame(
    results,
    columns=["Model","Accuracy","Precision","Recall","F1","ROC-AUC","CV F1"]
).sort_values(by="ROC-AUC", ascending=False)

print("\n==================== RESULTS ====================")
print(results_df)

print("\nBEST MODEL:", best_name)



==================== RESULTS ====================
                 Model  Accuracy  Precision    Recall        F1   ROC-AUC  \
0  Logistic Regression  0.806955   0.658385  0.566845  0.609195  0.841585   
4        Random Forest  0.792051   0.639175  0.497326  0.559398  0.825852   
2          Naive Bayes  0.655784   0.426877  0.866310  0.571933  0.809254   
5                  SVM  0.792761   0.644366  0.489305  0.556231  0.796050   
1                  KNN  0.747339   0.525281  0.500000  0.512329  0.771604   
3        Decision Tree  0.740951   0.512535  0.491979  0.502046  0.660957   

      CV F1  
0  0.597957  
4  0.546219  
2  0.579322  
5  0.576485  
1  0.537816  
3  0.500344  

BEST MODEL: Logistic Regression


In [15]:
# SAVE FINAL MODEL
# =========================
joblib.dump(best_model, "best_onehot_model.pkl")

print("\nModel + Columns saved successfully")


Model + Columns saved successfully


In [16]:
model = joblib.load("best_onehot_model.pkl")
print(model.named_steps["model"].n_features_in_)

30


In [17]:
df["Churn"].value_counts()

,count
Churn,
0,5174
1,1869
